# 00 - Setup e Configuração do Ambiente Unificado
**Squad 2 — Real Time for Business | Unificação Dupla 1 & Dupla 5**  
**Integrantes:** Zaiden Emiliano Segundo Seleme & Lucas Sousa Santos Oliveira  
**Tabelas de Escopo:**
1. `ecommerce_produtos`
2. `ecommerce_categorias`
3. `ecommerce_rastreamento_entregas`
4. `ecommerce_enderecos`

**Branch:** `feat/squad2-dupla1e5`

---

### Objetivos:
1. Instalação automatizada das bibliotecas essenciais no ambiente Databricks Serverless.
2. Localização dinâmica e carregamento seguro das credenciais via `python-dotenv` com busca hierárquica multi-nível.
3. Validação das variáveis de ambiente para **Azure Data Lake Storage Gen2 (ADLS Gen2)** e **Azure SQL Server**.
4. Configuração das credenciais OAuth FQDN para o conector Spark ABFS (`fs.azure.account.*`).
5. Mapeamento centralizado de caminhos no Data Lake diretamente na **raiz do contêiner `squad2`** (`/bronze/`, `/silver/`, `/quarantine/`, `/gold/`, `/metadata/`).
6. Teste de conectividade eager com o ADLS Gen2 utilizando leitor resiliente para contornar limitações de schemas (`TIMESTAMP_NANOS` e `INT32 Null`).

In [ ]:
# 1. Instalação das bibliotecas no escopo da sessão Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity pandas pyarrow pyodbc --quiet

In [ ]:
# 2. Reinicialização segura do interpretador Python para carregar novos pacotes
dbutils.library.restartPython()

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# Padronização de TimeZone Oficial da Squad 2 (Horário de Brasília - UTC-3)
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

# 3. Localização dinâmica e carga do arquivo .env com busca resiliente
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed",
        "/Workspace/Repos/estagio_ed"
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Arquivo .env carregado com sucesso de: {dotenv_path}")
else:
    load_dotenv(override=True)
    print("Tentativa de carregar .env a partir das variáveis de ambiente padrão.")

# Leitura e validação das variáveis do ADLS Gen2
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

# Leitura e validação das variáveis do Azure SQL Server
sql_host = os.getenv("SQL_HOST", "srv-database-intership.database.windows.net")
sql_database = os.getenv("SQL_DATABASE", "internshipDatabase")
sql_user = os.getenv("SQL_USERNAME", "estagiario_user")
sql_password = os.getenv("SQL_PASSWORD")

assert client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"
assert sql_password, "SQL_PASSWORD não configurado no .env"

print("\n=== Status das Credenciais ADLS Gen2 ===")
print(f"  Storage Account: {storage_account}")
print(f"  Client ID:       {'[CONFIGURADO]' if client_id else '[PENDENTE]'}")
print(f"  Tenant ID:       {'[CONFIGURADO]' if tenant_id else '[PENDENTE]'}")
print(f"  Client Secret:   {'[CONFIGURADO]' if client_secret else '[PENDENTE]'}")

print("\n=== Status das Credenciais SQL Server ===")
print(f"  SQL Host:        {sql_host}")
print(f"  SQL Database:    {sql_database}")
print(f"  SQL User:        {sql_user}")
print(f"  SQL Password:    {'[CONFIGURADO]' if sql_password else '[PENDENTE]'}")

In [ ]:
# 4. Configuração OAuth com FQDN do Storage Account para injeção granular no Spark
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# 5. Definição da malha de caminhos ABFSS diretamente na RAIZ do container squad2
base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net"

caminhos = {
    # Origens no container RAW (Landing Zone)
    "raw_produtos": f"{base_raw}/*/*/*/*/ecommerce_produtos.parquet",
    "raw_categorias": f"{base_raw}/*/*/*/*/ecommerce_categorias.parquet",
    "raw_rastreamento": f"{base_raw}/*/*/*/*/ecommerce_rastreamento.parquet",
    "raw_enderecos": f"{base_raw}/*/*/*/*/ecommerce_enderecos.parquet",
    
    # Camada Bronze (Raiz do container squad2)
    "bronze_produtos": f"{base_squad}/bronze/ecommerce_produtos",
    "bronze_categorias": f"{base_squad}/bronze/ecommerce_categorias",
    "bronze_rastreamento": f"{base_squad}/bronze/ecommerce_rastreamento_entregas",
    "bronze_enderecos": f"{base_squad}/bronze/ecommerce_enderecos",

    # Camada Silver (Raiz do container squad2)
    "silver_produtos": f"{base_squad}/silver/ecommerce_produtos",
    "silver_categorias": f"{base_squad}/silver/ecommerce_categorias",
    "silver_rastreamento": f"{base_squad}/silver/ecommerce_rastreamento_entregas",
    "silver_enderecos": f"{base_squad}/silver/ecommerce_enderecos",
    "silver_pedidos": f"{base_squad}/silver/ecommerce_pedidos",

    # Quarentena (Raiz do container squad2)
    "quarantine_produtos": f"{base_squad}/quarantine/ecommerce_produtos",
    "quarantine_categorias": f"{base_squad}/quarantine/ecommerce_categorias",
    "quarantine_rastreamento": f"{base_squad}/quarantine/ecommerce_rastreamento_entregas",
    "quarantine_enderecos": f"{base_squad}/quarantine/ecommerce_enderecos",

    # Camada Gold (Raiz do container squad2)
    "gold_dim_produtos": f"{base_squad}/gold/gold_dim_produtos",
    "gold_metricas_categorias": f"{base_squad}/gold/gold_metricas_categorias",
    "gold_pedidos_rota": f"{base_squad}/gold/gold_logistica_pedidos_rota",
    "gold_performance_transportadoras": f"{base_squad}/gold/gold_performance_transportadoras",
    "gold_distribuicao_geografica": f"{base_squad}/gold/gold_distribuicao_geografica_clientes",

    # Controle de Ingestão Unificado (Protegido com underscore)
    "metadata_control_log": f"{base_squad}/metadata/_ingestion_control_log"
}

print("Malha de caminhos na raiz do container squad2 configurada:")
for k, v in caminhos.items():
    print(f"  {k:32}: {v}")

In [ ]:
# 6. Teste de Validação de Conectividade Eager via Leitor Resiliente Serverless
import io
import pyarrow.parquet as pq
import pyarrow as pa
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential

schema_rastreamento_pa = pa.schema([
    ('id_rastreamento', pa.int64()),
    ('id_pedido_ecommerce', pa.int64()),
    ('codigo_rastreio', pa.string()),
    ('id_transportadora', pa.int64()),
    ('nome_transportadora', pa.string()),
    ('status_entrega', pa.string()),
    ('dt_evento', pa.timestamp('us')),
    ('observacao', pa.string()),
])

schema_enderecos_pa = pa.schema([
    ('id_endereco', pa.int64()),
    ('id_cliente', pa.int64()),
    ('logradouro', pa.string()),
    ('numero', pa.int64()),
    ('complemento', pa.string()),
    ('bairro', pa.string()),
    ('cep', pa.string()),
    ('cidade', pa.string()),
    ('estado', pa.string()),
    ('latitude', pa.float64()),
    ('longitude', pa.float64()),
    ('apelido', pa.string()),
    ('is_principal', pa.bool_()),
])

def ler_parquet_raw_resiliente(nome_arquivo_alvo, schema_pa):
    try:
        cred = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
        service = DataLakeServiceClient(account_url=f"https://{storage_account}.dfs.core.windows.net", credential=cred)
        fs = service.get_file_system_client("raw")
        
        paths = fs.get_paths(path="real-time-data")
        arquivos = [p.name for p in paths if p.name.endswith(nome_arquivo_alvo)]
        
        tables = []
        for p in arquivos:
            fc = fs.get_file_client(p)
            data = fc.download_file().readall()
            t = pq.read_table(io.BytesIO(data))
            if t.num_rows == 0 or len(t.schema.names) == 0:
                continue
            if set(schema_pa.names).issubset(set(t.schema.names)):
                t = t.select(schema_pa.names)
            t_casted = t.cast(schema_pa)
            full_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/{p}"
            t_with_meta = t_casted.append_column("bronze_source_file", pa.array([full_path] * len(t_casted)))
            tables.append(t_with_meta)
            
        if not tables:
            return None
        combined = pa.concat_tables(tables)
        return spark.createDataFrame(combined.to_pandas())
    except Exception as e:
        print(f"  [AVISO] Conexão com RAW: {e}")
        return None

print("\n=== Teste de Conectividade com Origens e Destinos ===")

# Teste 1: Produtos e Categorias via Spark direto
try:
    df_prod_teste = spark.read.options(**adls_options).parquet(caminhos["raw_produtos"]).limit(5)
    df_cat_teste = spark.read.options(**adls_options).parquet(caminhos["raw_categorias"]).limit(5)
    print(f"[SUCESSO] Origens de Produtos e Categorias acessíveis via Spark ABFS.")
except Exception as e_g1:
    print(f"[AVISO] Produtos/Categorias ainda não disponíveis em RAW ({e_g1.__class__.__name__}).")

# Teste 2: Rastreamento e Endereços via Leitor Resiliente
try:
    df_rast_teste = ler_parquet_raw_resiliente("ecommerce_rastreamento.parquet", schema_rastreamento_pa)
    df_end_teste = ler_parquet_raw_resiliente("ecommerce_enderecos.parquet", schema_enderecos_pa)
    print(f"[SUCESSO] Origens de Rastreamento ({df_rast_teste.count() if df_rast_teste else 0} linhas) e Endereços ({df_end_teste.count() if df_end_teste else 0} linhas) acessíveis.")
except Exception as e_g5:
    print(f"[AVISO] Rastreamento/Endereços ainda não disponíveis em RAW ({e_g5}).")

def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

print("\nStatus das Tabelas Delta na raiz de squad2:")
print(f"  Bronze Produtos:     {'[EXISTENTE]' if tabela_delta_existe(caminhos['bronze_produtos']) else '[NOVO]'}")
print(f"  Bronze Categorias:   {'[EXISTENTE]' if tabela_delta_existe(caminhos['bronze_categorias']) else '[NOVO]'}")
print(f"  Bronze Rastreamento: {'[EXISTENTE]' if tabela_delta_existe(caminhos['bronze_rastreamento']) else '[NOVO]'}")
print(f"  Bronze Endereços:    {'[EXISTENTE]' if tabela_delta_existe(caminhos['bronze_enderecos']) else '[NOVO]'}")

print("\nSetup Unificado da Squad 2 concluído com sucesso!")